# Process the goggles-disparity session

Builds the `DisparityDevice` (disparity-conditioned), runs
`DisparitySessionAnalyzer.analyze_session_stats`, and saves
`{session_name}_dataframe.pkl` next to the session data. That df is **long**
(one row per cell x disparity); `CrossExperimentAnalysis` collapses it to
reliable-in-any-disparity. It is what
`monocular-disparity-distance-comparison.ipynb` reads for this experiment via
`CrossExperimentAnalysis.from_devices(R, M, D)`.

Run this in its own kernel with **only** `disparity-analysis` on `sys.path`.

In [7]:
import sys, os
import pandas as pd

base_dir = r"C:\Users\acostamendoza\projects\distance-project"
for p in (base_dir, base_dir + r"\disparity-analysis"):
    if p not in sys.path:
        sys.path.append(p)

from src.data.DisparityDevice import DisparityDevice
from src.analysis.DisparityAnalysis import DisparitySessionAnalyzer

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
mouse, date, experiment = "SAM045", "260902", "disparity"
D = DisparityDevice(mouse_name=mouse, record_date=date, experiment_name=experiment)
print("session:", D.session_name, "| disparities:", D.disparities, "| cells:", len(D.suite2p_id))

Creating directory: results
session: 260902_SAM045_disparity | disparities: [-15, -10, -5, 0, 5, 10, 15] | cells: 987


In [9]:
# load cached stats if present, else compute once (slow) and save the pkl the
# comparison notebook reads
pkl_path = os.path.join(D.data_dir, f"{D.session_name}_dataframe.pkl")
if os.path.exists(pkl_path):
    stats_df = pd.read_pickle(pkl_path)
    print(f"Loaded cached stats: {pkl_path}")
else:
    stats_df = DisparitySessionAnalyzer(D).analyze_session_stats(n_iterations=200)
    DisparitySessionAnalyzer(D).save_dataframe(stats_df)   # -> pkl_path
stats_df.head()

Saved DataFrame to: D:/data-distance-coding-project/processed-data\260902_SAM045_disparity\260902_SAM045_disparity_dataframe.pkl


,cell,session_name,disparity,reliable,mean_corr,rf,rf_value,rf_deg
0,0,260902_SAM045_disparity,-15,False,0.592767,15,0.636046,-14.745763
1,0,260902_SAM045_disparity,-10,False,0.157240,35,0.344500,5.593220
2,0,260902_SAM045_disparity,-5,False,-0.199212,22,0.564661,-7.627119
3,0,260902_SAM045_disparity,0,False,0.023045,34,0.114203,4.576271
4,0,260902_SAM045_disparity,5,False,0.056943,47,0.112313,17.796610
